# Projeto Final — Servo de Posição com Motor CC

**Laboratório de Controle Automático — Ifes Guarapari**

| | |
|---|---|
| **Aluno(a):** | *(seu nome)* |
| **Bancada / colega:** | *(nº da bancada / nome do colega)* |
| **Metas de projeto (Rota A):** | *(ζ = …, ts = … — diferentes das do colega)* |
| **Opcionais feitos:** | *(Rota B · refinamento do ZN · back-calculation · stick-slip)* |

> **Como usar este modelo:** copie este arquivo para `projeto_<seunome>.ipynb`, na mesma pasta de
> `ferramentas_projeto.py`, com os CSVs numa subpasta `dados/` (nomes na Seção 7 do enunciado).
> Preencha as células marcadas com `...` e **escreva a discussão** de cada seção. Entregue o
> notebook **executado**. O enunciado completo está em `projeto_final_motor_cc.md`.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import control as ct
from scipy.optimize import minimize

try:
    from ferramentas_projeto import *
except ImportError:                                  # notebook fora da pasta projeto_final/
    sys.path.insert(0, os.path.abspath(".."))
    from ferramentas_projeto import *

DIR_DADOS = "dados"
arq = lambda nome: os.path.join(DIR_DADOS, nome)
Ts = 0.02
s = ct.tf("s")
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})
print("control", ct.__version__)

## S — Preparação: teste de sentido (MODO 1, `PWM_MA = 100`)

In [ ]:
teste = carregar_csv(arq("teste.csv"))
plt.plot(teste["t"], teste["angle"]); plt.xlabel("tempo (s)"); plt.ylabel("ângulo (graus)")
plt.title("O ângulo tem que CRESCER com PWM positivo"); plt.show()

## E0 — Calibração contas → graus (MODO 0)

In [ ]:
cal = carregar_csv(arq("calibracao.csv"))
N_VOLTAS = 5
contas_finais = ...                 # 5ª coluna (u_pid) no último instante
CPG = ...                           # CONTAS_POR_GRAU
print("CONTAS_POR_GRAU =", CPG, "| resolução =", 1 / CPG, "grau/conta")

**Discussão E0:** *(contas por volta esperadas pela folha de dados × medidas; por que o firmware conta 1×)*

## E1 — Modelo de velocidade por PRBS (= Trabalho 1)

In [ ]:
prbs = carregar_csv(arq("prbs.csv"))
t1, u1, y1 = prbs["t"] - prbs["t"][0], prbs["u"], prbs["y"]

def simula_fopdt(p, t, u, com_offset):
    K, tau, L = p[:3]
    u0 = p[3] if com_offset else 0.0
    ...                              # atraso = deslocar a entrada de round(L/Ts) amostras; entrada (u - u0)
    return ys

def custo(p, com_offset):
    ...                              # erro quadrático médio; penalize parâmetros fisicamente impossíveis

fit_sem = minimize(custo, [0.1, 0.1, 0.02], args=(False,), method="Nelder-Mead")
fit_com = minimize(custo, [0.1, 0.1, 0.02, 20], args=(True,), method="Nelder-Mead")
K, tau, L, u0 = fit_com.x

In [ ]:
# gráfico: medido × modelo sem desvio × modelo com desvio (um trecho de alguns segundos)
...

In [ ]:
# Padé de 1ª e de 10ª ordem (Passo 5 do Trabalho 1): monte os dois modelos, simule com a mesma
# entrada (u1 - u0) e compare o EQM com o do FOPDT
...

**Discussão E1:** *(escolha de Tb; comparação dos dois ajustes e dos modelos com Padé; significado de $u_0$; origem do atraso $L$ e o que dele existe na malha de posição)*

## E2 — Modelo de posição, zona morta e atrito estático
### E2.1–E2.2 Inclinações, $K_m$, $F_c^\pm$ e $k'$

In [ ]:
ens = {A: carregar_csv(arq(n + ".csv")) for n, A in
       [("ma_p100", 100), ("ma_p200", 200), ("ma_m100", -100), ("ma_m200", -200)]}
a = {}
for A, d in ens.items():
    m = (d["t"] >= 1.5) & (d["t"] < 2.5)
    a[A] = ...                       # inclinação da reta (np.polyfit)

Km_p = ...;  Fc_p = ...
Km_n = ...;  Fc_n = ...
Km = ...
k_linha, k_linha_teo = Km / K, 1 / (Ts * CPG)

### E2.3 Atrito estático (teste de partida)

In [ ]:
FS_P = ...      # menor PWM positivo que tirou o eixo do repouso
FS_N = ...      # idem, negativo (em módulo)

### E2.4 Gêmeo digital e validação

In [ ]:
GEMEO = dict(Km=Km, tau=tau, L=L, Fc=Fc_p, Fc_neg=Fc_n, Fs=FS_P, Fs_neg=FS_N, cpg=CPG)
for A in (100, -200):
    sim = simula_firmware(GEMEO, modo="malha_aberta", pwm_ma=A, ma_ini=0.5, ma_fim=2.5)
    ...                              # sobreponha sim["angle"] e ens[A]["angle"]

**Tabela do Marco 1:** *(CPG, K, τ, L, u₀, Km±, Fc±, Fs±, k' × k'teo)*

**Discussão E2:** *(simetria/assimetria; F_s × F_c; qualidade do gêmeo)*

## E3 — Projeto do controlador

$$C_y(s) = k_p + \frac{k_i}{s} + k_d\frac{Ns}{s+N}, \qquad T(s) = \frac{G_p(s)(k_p + k_i/s)}{1 + G_p(s)C_y(s)}, \qquad L_{ef} = L + T_s/2$$

In [ ]:
N_FILT = ...                                  # justifique na E4
G = Km / (s * (tau * s + 1))
Pd = ct.tf(*ct.pade(L + Ts / 2, 3))

def analisa(nome, kp, ki, kd, N=N_FILT):
    Cy = kp + ki / s + kd * N * s / (s + N)
    gm, pm, wg, wc = ct.margin(Cy * G * Pd)
    T = ct.feedback(G * Pd, Cy) * (kp + ki / s)
    tt = np.linspace(0, 6, 3001)
    _, y = ct.step_response(90 * T, tt)
    m = metricas_degrau(np.r_[-1e-3, tt], np.r_[0, np.full_like(tt, 90)], np.r_[0, y], 0, 6)
    print(f"{nome}: kp={kp:.3f} ki={ki:.3f} kd={kd:.3f} | GM={20*np.log10(gm):.1f} dB "
          f"PM={pm:.1f}° wc={wc:.2f} rad/s | linear: Mp={m['Mp']:.1f}% ts={m['ts']:.2f}s")
    return T

### Iteração 1 — Rota A, projeto para a especificação
*(polo desejado $s_d$; $\angle G_p(s_d)$ com o atraso; precisa de avanço? ganho pela condição de módulo; zero do PI. Resultado linear e no gêmeo — ver E5; diagnóstico: quanto vale $F_s/k_p$?)*

In [ ]:
kp1, ki1, kd1 = ...
T1 = analisa("Iteração 1", kp1, ki1, kd1)

### Rota A — iteração final
*(novas metas e por quê; contas passo a passo — escreva-as aqui em LaTeX; confira no Python)*

In [ ]:
...
kpF, kiF, kdF = ...
TF = analisa("Final", kpF, kiF, kdF)

In [ ]:
# LGR com s_d marcado (obrigatório).  Se fizer a Rota B (opcional): ct.bode_plot(Cy*G*Pd, display_margins=True)
...

### Rota C — relé + Ziegler-Nichols (verificação; refinamento opcional)

In [ ]:
rele = carregar_csv(arq("rele.csv"))
d_rele, h_rele = 120, 1.0
...                                           # a (semi-amplitude), Pu (período médio), Ku
# ZN-PID e suas margens:
# analisa("ZN puro", 0.6*Ku, 0.6*Ku/(Pu/2), 0.6*Ku*Pu/8)

**Tabela do Marco 2:** *(iteração 1, Rota A final, ZN: ganhos, GM, PM, ωc, Mp e ts lineares)*

**Discussão E3:** *(comparação com o projeto do colega e com o ZN; por que o ZN puro se comporta assim nesta planta)*

## E4 — Realização digital
*(Ts frente a ωc; ZOH ≈ atraso de Ts/2; o que acontece com L = 84 ms; escolha de N (teste 10, 30, 50); ruído de uma conta no termo derivativo; equações de recorrência do firmware)*

In [ ]:
lin = dict(Km=Km, tau=tau, L=L)                # modelo LINEAR (sem atrito, sem quantização)
disc = simula_firmware(lin, dict(kp=kpF, ki=kiF, kd=kdF, N=N_FILT), perfil=[(0, 0), (0.02, 90)], T_fim=5.02)
...                                            # sobreponha ao degrau contínuo de 90*TF
linhas_firmware(kpF, kiF, kdF, N_FILT)

## E5 — Gêmeo digital: atrito, compensação e robustez

In [ ]:
CTRL = dict(kp=kpF, ki=kiF, kd=kdF, N=N_FILT, pwm_max=255, aw=1, zm_comp=0, e_morto=0.0)
sim = simula_firmware(GEMEO, CTRL)
imprime_metricas(metricas_ensaio(sim), "gêmeo — sem compensação")
plota_ensaio(sim, titulo="gêmeo digital"); plt.show()
# teste também: zm_comp > 0 com e_morto = 0, e zm_comp > 0 com e_morto > 0 (olhe o PWM!)
...

In [ ]:
# Robustez: atrito sorteado em ±15 %
rng = np.random.default_rng(0)
def sorteia_planta():
    p = dict(GEMEO)
    for k in ("Fc", "Fs", "Fc_neg", "Fs_neg"):
        p[k] = GEMEO[k] * (1 + 0.15 * rng.uniform(-1, 1))
    p["Fs"], p["Fs_neg"] = max(p["Fs"], p["Fc"]), max(p["Fs_neg"], p["Fc_neg"])
    return p
plantas = [sorteia_planta() for _ in range(20)]

def aprovadas(ctrl):
    ok = 0
    for p in plantas:
        ms = metricas_ensaio(simula_firmware(p, ctrl))
        ok += all(m["Mp"] <= 15 and m["ts"] <= 2.5 and abs(m["ess"]) < 2 for m in ms)
    return ok

# compare pelo menos: sem compensação · a sua configuração final · a final com ki bem maior
...

**Valores finais para o firmware:** `kp = …`, `ki = …`, `kd = …`, `N_FILT = …`, `ZM_COMP = …`, `E_MORTO = …`

**Discussão E5:** *(o que cada ferramenta mudou; conclusão da robustez sobre o integrador)*

## E6 — Bancada: especificações 1–3

In [ ]:
nominais = [carregar_csv(arq(f"mf_nominal_{i}.csv")) for i in (1, 2, 3)]
for i, d in enumerate(nominais, 1):
    imprime_metricas(metricas_ensaio(d), f"\nBANCADA — repetição {i}")
CTRL_FINAL = dict(CTRL, zm_comp=..., e_morto=...)      # os valores que foram para o firmware
sim_final = simula_firmware(GEMEO, CTRL_FINAL)
imprime_metricas(metricas_ensaio(sim_final), "\nGÊMEO — previsão")
plota_ensaio(sim_final, nominais[0], rotulos=["gêmeo", "bancada (rep. 1)"]); plt.show()

**Discussão E6:** *(medido × linear × gêmeo; assimetria; dispersão; veredito das especificações 1–3 no pior caso)*

## E7 — Especificação 4: saturação e anti-windup

In [ ]:
PWM_SAT = 80
sat1, sat0 = carregar_csv(arq("mf_sat_aw1.csv")), carregar_csv(arq("mf_sat_aw0.csv"))
imprime_metricas(metricas_ensaio(sat1, pwm_max=PWM_SAT), "AW ligado")
imprime_metricas(metricas_ensaio(sat0, pwm_max=PWM_SAT), "\nAW desligado")
# gráfico: ângulo, pwm e u_pid com e sem anti-windup (zoom no degrau de 90° e no de 180°)
...

**Discussão E7:** *(onde o clamping congela o integrador; veredito da especificação 4)*

## E8 — Não linearidades

| Não linearidade | Onde aparece nos meus dados | Como foi tratada |
|---|---|---|
| Saturação do PWM | | |
| Zona morta / atrito de Coulomb | | |
| Atrito estático | | |
| Quantização do encoder | | |
| Atraso (atuador + ZOH) | | |

## Conclusão
*(o que faria diferente; limitações do modelo FOPDT + atrito; o que aprendeu com a comparação entre as rotas)*